In [2]:
# ============================================================
# IMPORTS
# ============================================================

import json
import random
import warnings
from pathlib import Path
import gc
import joblib

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
    classification_report
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# 02 — IDM Selector Comparison

This notebook compares feasible methods for selecting or combining IDM candidate predictions. The goal is to determine whether learned selectors can improve the IDM prior beyond the global IDM and simple regime-aware assignment. The strongest feasible selector is later used as an official IDM variant and as a possible hybrid prior.

In [3]:
# ============================================================
# CONFIG
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

DATA_DIR = PROJECT_DIR / "processed_data_final"

OUTPUT_DIR = DATA_DIR / "idm_selector_comparison_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BRAKE_THRESHOLD = -0.2
ACCEL_THRESHOLD = 0.2

REGIME_TAG = "symmetric"
THRESHOLD_TAG = f"{REGIME_TAG}_brake{abs(BRAKE_THRESHOLD):.1f}_accel{ACCEL_THRESHOLD:.1f}".replace(".", "p")

RUN_DIR = OUTPUT_DIR / f"idm_selector_comparison_{THRESHOLD_TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Run dir:", RUN_DIR)
print("Brake threshold:", BRAKE_THRESHOLD)
print("Accel threshold:", ACCEL_THRESHOLD)

assert DATA_DIR.exists(), f"Data directory does not exist: {DATA_DIR}"

Data dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_symmetric_brake0p2_accel0p2
Brake threshold: -0.2
Accel threshold: 0.2


In [4]:
# ============================================================
# LOAD PRE-SPLIT DATASETS
# Memory-safe loading for symmetric selector comparison
# ============================================================

required_files = [
    DATA_DIR / "episode_tracks_train.csv",
    DATA_DIR / "episode_tracks_val.csv",
    DATA_DIR / "episode_tracks_test.csv",
]

for p in required_files:
    assert p.exists(), f"Missing required file: {p}"

needed_cols = [
    "recording_id",
    "id",
    "unique_id",
    "episode_id",
    "frame",
    "xVelocity",
    "precedingId",
    "precedingXVelocity",
    "xAcceleration",
    "dhw",
    "split",
]

dtype_map = {
    "recording_id": "int32",
    "id": "int32",
    "frame": "int32",
    "precedingId": "float32",
    "xVelocity": "float32",
    "precedingXVelocity": "float32",
    "xAcceleration": "float32",
    "dhw": "float32",
}


def read_episode_tracks_minimal(path, chunksize=500_000):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in needed_cols if c in header]
    dtype_use = {k: v for k, v in dtype_map.items() if k in usecols}

    chunks = []

    for i, chunk in enumerate(
        pd.read_csv(
            path,
            usecols=usecols,
            dtype=dtype_use,
            chunksize=chunksize,
            low_memory=False,
        )
    ):
        for c in ["unique_id", "episode_id", "split"]:
            if c in chunk.columns:
                chunk[c] = chunk[c].astype("category")

        chunks.append(chunk)

        if (i + 1) % 5 == 0:
            print(f"{path.name}: loaded {(i + 1) * chunksize:,} rows...")
            gc.collect()

    df = pd.concat(chunks, ignore_index=True)

    del chunks
    gc.collect()

    for c in ["unique_id", "episode_id", "split"]:
        if c in df.columns:
            df[c] = df[c].astype("category")

    print(
        f"Loaded {path.name}: {df.shape}, "
        f"{df.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
    )

    return df


episode_tracks_train = read_episode_tracks_minimal(DATA_DIR / "episode_tracks_train.csv")
episode_tracks_val   = read_episode_tracks_minimal(DATA_DIR / "episode_tracks_val.csv")
episode_tracks_test  = read_episode_tracks_minimal(DATA_DIR / "episode_tracks_test.csv")

print("train:", episode_tracks_train.shape)
print("val  :", episode_tracks_val.shape)
print("test :", episode_tracks_test.shape)

episode_tracks_train.csv: loaded 2,500,000 rows...
episode_tracks_train.csv: loaded 5,000,000 rows...
episode_tracks_train.csv: loaded 7,500,000 rows...
episode_tracks_train.csv: loaded 10,000,000 rows...
Loaded episode_tracks_train.csv: (11094352, 11), 435.9 MB
episode_tracks_val.csv: loaded 2,500,000 rows...
Loaded episode_tracks_val.csv: (2370107, 11), 84.1 MB
episode_tracks_test.csv: loaded 2,500,000 rows...
Loaded episode_tracks_test.csv: (2368174, 11), 84.0 MB
train: (11094352, 11)
val  : (2370107, 11)
test : (2368174, 11)


In [5]:
# ============================================================
# FEATURE PREPARATION
# Memory-safe version
# ============================================================

def prepare_base_features(df):
    keep_cols = [
        "episode_id",
        "unique_id",
        "split",
        "frame",
        "xVelocity",
        "precedingId",
        "precedingXVelocity",
        "xAcceleration",
        "dhw",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    out = df[keep_cols].copy()

    out["v_ego"] = out["xVelocity"].abs().astype(np.float32)
    out["v_leader"] = out["precedingXVelocity"].abs().astype(np.float32)
    out["a_true"] = out["xAcceleration"].astype(np.float32)

    no_leader = (
        out["precedingId"].isna()
        | (out["precedingId"] <= 0)
        | out["precedingXVelocity"].isna()
        | out["dhw"].isna()
    )

    out["delta_v"] = (out["v_ego"] - out["v_leader"]).astype(np.float32)
    out["gap"] = out["dhw"].clip(lower=0.1).astype(np.float32)

    out.loc[no_leader, "gap"] = np.float32(1e6)
    out.loc[no_leader, "delta_v"] = np.float32(0.0)
    out.loc[no_leader, "v_leader"] = out.loc[no_leader, "v_ego"]

    out["no_leader"] = no_leader.astype("int8")

    final_cols = [
        "episode_id",
        "unique_id",
        "split",
        "frame",
        "v_ego",
        "v_leader",
        "delta_v",
        "gap",
        "a_true",
        "no_leader",
    ]

    out = (
        out[final_cols]
        .sort_values(["episode_id", "frame"])
        .reset_index(drop=True)
    )

    return out


train_base = prepare_base_features(episode_tracks_train)
val_base   = prepare_base_features(episode_tracks_val)
test_base  = prepare_base_features(episode_tracks_test)

del episode_tracks_train, episode_tracks_val, episode_tracks_test
gc.collect()

print(train_base[[
    "episode_id", "frame", "v_ego", "v_leader",
    "delta_v", "gap", "a_true", "no_leader"
]].head())

print("\nBase shapes:")
print("train_base:", train_base.shape)
print("val_base  :", val_base.shape)
print("test_base :", test_base.shape)

  episode_id  frame      v_ego   v_leader   delta_v        gap  a_true  \
0          1  21549  41.189999  43.099998 -1.910000  29.230000   -0.39   
1          1  21550  41.200001  43.130001 -1.930000  29.459999   -0.39   
2          1  21551  41.220001  43.150002 -1.930000  29.639999   -0.40   
3          1  21552  41.230000  43.180000 -1.950001  29.780001   -0.41   
4          1  21553  41.250000  43.200001 -1.950001  29.900000   -0.42   

   no_leader  
0          0  
1          0  
2          0  
3          0  
4          0  

Base shapes:
train_base: (11094352, 10)
val_base  : (2370107, 10)
test_base : (2368174, 10)


In [6]:
# ============================================================
# IDM ACCELERATION
# ============================================================

def idm_acceleration(v, delta_v, s, params):
    v0 = max(float(params["v0"]), 1e-6)
    T = float(params["T"])
    s0 = float(params["s0"])
    a = max(float(params["a"]), 1e-6)
    b = max(float(params["b"]), 1e-6)
    delta = float(params["delta"])

    v = np.maximum(v, 0.0)
    s = np.maximum(s, 0.1)

    s_star = s0 + np.maximum(
        0.0,
        v * T + (v * delta_v) / (2.0 * np.sqrt(a * b))
    )

    acc = a * (
        1.0
        - (v / v0) ** delta
        - (s_star / s) ** 2
    )

    acc = np.clip(acc, -8.0, 4.0)

    return acc

In [7]:
# ============================================================
# FINAL IDM PARAMETER SETS
# ============================================================

best_global_params = {"v0": 40.0, "T": 0.5,"s0": 0.5, "a": 0.4,"b": 5.0,"delta": 4.0}

best_params_braking = {'v0': 1.5, 'T': 0.3, 's0': 5.0, 'a': 0.1, 'b': 26.0, 'delta': 0.5}

best_params_accel = {'v0': 115.0, 'T': 0.1, 's0': 0.05, 'a': 0.4, 'b': 22.0, 'delta': 14.0}

best_params_steady = {'v0': 44.0, 'T': 0.3, 's0': 0.1, 'a': 0.1, 'b': 8.0, 'delta': 2.0}


In [8]:
# ============================================================
# SELECTOR FEATURE ENGINEERING
# ============================================================

def add_selector_features(df):
    out = df.copy()

    out["time_headway"] = out["gap"] / np.maximum(out["v_ego"], 0.1)
    out["closing_speed"] = np.maximum(out["delta_v"], 0.0)
    out["opening_speed"] = np.maximum(-out["delta_v"], 0.0)
    out["abs_delta_v"] = np.abs(out["delta_v"])
    out["inv_ttc"] = out["closing_speed"] / np.maximum(out["gap"], 0.1)
    out["log_gap"] = np.log1p(np.minimum(out["gap"], 500.0))

    return out


selector_cols = [
    "v_ego",
    "v_leader",
    "delta_v",
    "gap",
    "time_headway",
    "closing_speed",
    "opening_speed",
    "abs_delta_v",
    "inv_ttc",
    "log_gap"
]

train_sel = add_selector_features(train_base)
val_sel   = add_selector_features(val_base)
test_sel  = add_selector_features(test_base)

print("Selector features:")
print(selector_cols)

print(train_sel[selector_cols + ["a_true"]].head())

Selector features:
['v_ego', 'v_leader', 'delta_v', 'gap', 'time_headway', 'closing_speed', 'opening_speed', 'abs_delta_v', 'inv_ttc', 'log_gap']
       v_ego   v_leader   delta_v        gap  time_headway  closing_speed  \
0  41.189999  43.099998 -1.910000  29.230000      0.709638            0.0   
1  41.200001  43.130001 -1.930000  29.459999      0.715048            0.0   
2  41.220001  43.150002 -1.930000  29.639999      0.719068            0.0   
3  41.230000  43.180000 -1.950001  29.780001      0.722290            0.0   
4  41.250000  43.200001 -1.950001  29.900000      0.724848            0.0   

   opening_speed  abs_delta_v  inv_ttc   log_gap  a_true  
0       1.910000     1.910000      0.0  3.408835   -0.39  
1       1.930000     1.930000      0.0  3.416414   -0.39  
2       1.930000     1.930000      0.0  3.422306   -0.40  
3       1.950001     1.950001      0.0  3.426865   -0.41  
4       1.950001     1.950001      0.0  3.430756   -0.42  


In [9]:
# ============================================================
# TRUE REGIME LABELS
# ============================================================

def add_true_regime(df, true_col="a_true"):
    out = df.copy()

    out["true_regime"] = np.where(
        out[true_col] < BRAKE_THRESHOLD,
        "braking",
        np.where(
            out[true_col] > ACCEL_THRESHOLD,
            "accelerating",
            "steady"
        )
    )

    return out


train_sel = add_true_regime(train_sel)
val_sel   = add_true_regime(val_sel)
test_sel  = add_true_regime(test_sel)

print("Train regime counts:")
print(train_sel["true_regime"].value_counts())

print("\nValidation regime counts:")
print(val_sel["true_regime"].value_counts())

print("\nTest regime counts:")
print(test_sel["true_regime"].value_counts())

Train regime counts:
true_regime
steady          6522921
accelerating    3182947
braking         1388484
Name: count, dtype: int64

Validation regime counts:
true_regime
steady          1385845
accelerating     685291
braking          298971
Name: count, dtype: int64

Test regime counts:
true_regime
steady          1411748
accelerating     672329
braking          284097
Name: count, dtype: int64


In [10]:
# ============================================================
# ADD ALL IDM CANDIDATE PREDICTIONS
# ============================================================

def add_all_idm_candidates(df):
    out = df.copy()

    out["a_idm_global"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_global_params
    ).astype(np.float32)

    out["a_idm_braking"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_params_braking
    ).astype(np.float32)

    out["a_idm_accel"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_params_accel
    ).astype(np.float32)

    out["a_idm_steady"] = idm_acceleration(
        out["v_ego"].to_numpy(np.float32),
        out["delta_v"].to_numpy(np.float32),
        out["gap"].to_numpy(np.float32),
        best_params_steady
    ).astype(np.float32)

    return out


train_sel = add_all_idm_candidates(train_sel)
val_sel   = add_all_idm_candidates(val_sel)
test_sel  = add_all_idm_candidates(test_sel)

candidate_cols = [
    "a_idm_global",
    "a_idm_braking",
    "a_idm_accel",
    "a_idm_steady"
]

print(test_sel[candidate_cols].describe())

       a_idm_global  a_idm_braking   a_idm_accel  a_idm_steady
count  2.368174e+06   2.368174e+06  2.368174e+06  2.368174e+06
mean   1.514028e-01  -3.437748e-01  3.854747e-01  2.451495e-02
std    3.399127e-01   9.580187e-02  4.332486e-02  1.205679e-01
min   -8.000000e+00  -3.439968e+00 -2.342776e+00 -7.408484e+00
25%    1.081494e-01  -3.830971e-01  3.884812e-01  2.621355e-02
50%    2.444118e-01  -3.412948e-01  3.985026e-01  5.244933e-02
75%    3.297479e-01  -2.999975e-01  3.999984e-01  6.959921e-02
max    3.999899e-01   8.341831e-02  4.000000e-01  9.999982e-02


In [11]:
# ============================================================
# IDM METRIC HELPERS
# ============================================================

def compute_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    corr = np.nan
    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Bias": float(np.mean(y_pred - y_true)),
        "Corr": corr
    }


def evaluate_idm_prediction(df, pred_col="a_idm_selected", model_name="model"):
    overall = compute_regression_metrics(df["a_true"], df[pred_col])
    overall["Model"] = model_name

    regime_rows = []

    for regime, g in df.groupby("true_regime"):
        m = compute_regression_metrics(g["a_true"], g[pred_col])
        m["Model"] = model_name
        m["Regime"] = regime
        m["Count"] = len(g)
        regime_rows.append(m)

    overall_df = pd.DataFrame([overall])
    regime_df = pd.DataFrame(regime_rows)

    regime_order = ["accelerating", "braking", "steady"]
    regime_df["Regime"] = pd.Categorical(
        regime_df["Regime"],
        categories=regime_order,
        ordered=True
    )

    regime_df = regime_df.sort_values("Regime").reset_index(drop=True)

    return overall_df, regime_df

In [12]:
# ============================================================
# SELECTED IDM PRIOR FROM LABEL
# ============================================================

def selected_idm_from_label(df, label_col):
    out = df.copy()

    out["a_idm_selected"] = np.nan

    out.loc[out[label_col] == "global", "a_idm_selected"] = out.loc[
        out[label_col] == "global", "a_idm_global"
    ]

    out.loc[out[label_col] == "braking", "a_idm_selected"] = out.loc[
        out[label_col] == "braking", "a_idm_braking"
    ]

    out.loc[out[label_col] == "accelerating", "a_idm_selected"] = out.loc[
        out[label_col] == "accelerating", "a_idm_accel"
    ]

    out.loc[out[label_col] == "steady", "a_idm_selected"] = out.loc[
        out[label_col] == "steady", "a_idm_steady"
    ]

    if out["a_idm_selected"].isna().any():
        missing = out.loc[out["a_idm_selected"].isna(), label_col].unique()
        raise ValueError(f"Some labels could not be mapped: {missing}")

    return out

In [13]:
# ============================================================
# BASELINE IDM REFERENCES
# ============================================================

baseline_overall_rows = []
baseline_regime_rows = []

# 1. Global IDM
global_tmp = test_sel.copy()
global_tmp["a_idm_selected"] = global_tmp["a_idm_global"]

overall_df, regime_df = evaluate_idm_prediction(
    global_tmp,
    pred_col="a_idm_selected",
    model_name="Global_IDM"
)

baseline_overall_rows.append(overall_df)
baseline_regime_rows.append(regime_df)

# 2. True-regime oracle IDM
oracle_tmp = test_sel.copy()
oracle_tmp["selector_label"] = oracle_tmp["true_regime"]
oracle_tmp = selected_idm_from_label(oracle_tmp, "selector_label")

overall_df, regime_df = evaluate_idm_prediction(
    oracle_tmp,
    pred_col="a_idm_selected",
    model_name="TrueRegime_Oracle_IDM"
)

baseline_overall_rows.append(overall_df)
baseline_regime_rows.append(regime_df)

# 3. Ideal best-candidate IDM
def add_best_idm_label(df):
    out = df.copy()

    candidates = {
        "global": "a_idm_global",
        "braking": "a_idm_braking",
        "accelerating": "a_idm_accel",
        "steady": "a_idm_steady"
    }

    errors = np.column_stack([
        np.abs(out["a_true"].values - out[col].values)
        for col in candidates.values()
    ])

    best_idx = np.argmin(errors, axis=1)
    label_names = np.array(list(candidates.keys()))

    out["best_idm_label"] = label_names[best_idx]
    out["best_idm_error"] = errors[np.arange(len(out)), best_idx]

    return out


train_best = add_best_idm_label(train_sel)
val_best   = add_best_idm_label(val_sel)
test_best  = add_best_idm_label(test_sel)

ideal_best_tmp = test_best.copy()
ideal_best_tmp["selector_label"] = ideal_best_tmp["best_idm_label"]
ideal_best_tmp = selected_idm_from_label(ideal_best_tmp, "selector_label")

overall_df, regime_df = evaluate_idm_prediction(
    ideal_best_tmp,
    pred_col="a_idm_selected",
    model_name="Ideal_BestCandidate_IDM"
)

baseline_overall_rows.append(overall_df)
baseline_regime_rows.append(regime_df)

baseline_overall_df = pd.concat(baseline_overall_rows, ignore_index=True)
baseline_regime_df = pd.concat(baseline_regime_rows, ignore_index=True)

print("=== BASELINE IDM REFERENCES ===")
print(
    baseline_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)

print("\nBest-IDM label counts on test:")
print(test_best["best_idm_label"].value_counts())

=== BASELINE IDM REFERENCES ===
                  Model      MAE     RMSE    MedAE        R2      Bias      Corr
Ideal_BestCandidate_IDM 0.108056 0.195589 0.069797  0.637827 -0.006277  0.804899
  TrueRegime_Oracle_IDM 0.129256 0.214151 0.092586  0.565826  0.001369  0.753399
             Global_IDM 0.326738 0.486738 0.246854 -1.242930  0.061808 -0.053977

Best-IDM label counts on test:
best_idm_label
steady          1076162
accelerating     492715
global           445596
braking          353701
Name: count, dtype: int64


In [14]:
# ============================================================
# REGIME SELECTOR MODELS
# ============================================================

selector_models = {
    "DecisionTree_regime": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=500,
        random_state=SEED,
        class_weight="balanced"
    ),

    "RandomForest_regime": RandomForestClassifier(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=300,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced_subsample"
    ),

    "ExtraTrees_regime": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=200,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced"
    ),

    "HistGB_regime": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=SEED
    ),

    "KNN_regime": Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=51,
            weights="distance",
            n_jobs=-1
        ))
    ])
}

In [15]:
# ============================================================
# TRAIN REGIME SELECTORS AND EVALUATE DOWNSTREAM IDM
# ============================================================

MAX_TRAIN_ROWS_FOR_KNN = 250_000

regime_selector_overall_rows = []
regime_selector_regime_rows = []
regime_classification_rows = []

X_train_full = train_sel[selector_cols]
y_train_regime_full = train_sel["true_regime"]

X_test = test_sel[selector_cols]
y_test_regime = test_sel["true_regime"]

for name, clf in selector_models.items():
    print(f"\n==============================")
    print(f"Training selector: {name}")
    print(f"==============================")

    if name.startswith("KNN"):
        sample_idx = train_sel.sample(
            n=min(MAX_TRAIN_ROWS_FOR_KNN, len(train_sel)),
            random_state=SEED
        ).index

        X_train_use = train_sel.loc[sample_idx, selector_cols]
        y_train_use = train_sel.loc[sample_idx, "true_regime"]
    else:
        X_train_use = X_train_full
        y_train_use = y_train_regime_full

    clf.fit(X_train_use, y_train_use)

    pred_test = clf.predict(X_test)

    print("\nClassification report on test:")
    print(classification_report(y_test_regime, pred_test, digits=4))

    tmp = test_sel.copy()
    tmp["selector_label"] = pred_test
    tmp = selected_idm_from_label(tmp, "selector_label")

    overall_df, regime_df = evaluate_idm_prediction(
        tmp,
        pred_col="a_idm_selected",
        model_name=name
    )

    regime_selector_overall_rows.append(overall_df)
    regime_selector_regime_rows.append(regime_df)

    regime_classification_rows.append({
        "Model": name,
        "Accuracy": np.mean(pred_test == y_test_regime),
        "Pred_accelerating": np.sum(pred_test == "accelerating"),
        "Pred_braking": np.sum(pred_test == "braking"),
        "Pred_steady": np.sum(pred_test == "steady")
    })


regime_selector_overall_df = pd.concat(regime_selector_overall_rows, ignore_index=True)
regime_selector_regime_df = pd.concat(regime_selector_regime_rows, ignore_index=True)
regime_classification_df = pd.DataFrame(regime_classification_rows)

print("\n=== REGIME SELECTOR DOWNSTREAM IDM OVERALL RESULTS ===")
print(
    regime_selector_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


Training selector: DecisionTree_regime

Classification report on test:
              precision    recall  f1-score   support

accelerating     0.3875    0.5100    0.4404    672329
     braking     0.2239    0.6141    0.3281    284097
      steady     0.8050    0.4014    0.5357   1411748

    accuracy                         0.4577   2368174
   macro avg     0.4721    0.5085    0.4347   2368174
weighted avg     0.6167    0.4577    0.4837   2368174


Training selector: RandomForest_regime

Classification report on test:
              precision    recall  f1-score   support

accelerating     0.3950    0.5123    0.4461    672329
     braking     0.2520    0.5180    0.3391    284097
      steady     0.7685    0.4965    0.6033   1411748

    accuracy                         0.5036   2368174
   macro avg     0.4718    0.5090    0.4628   2368174
weighted avg     0.6005    0.5036    0.5270   2368174


Training selector: ExtraTrees_regime

Classification report on test:
              precision 

In [16]:
# ============================================================
# BEST-IDM SELECTOR MODELS
# ============================================================

best_selector_models = {
    "DecisionTree_bestIDM": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=500,
        random_state=SEED,
        class_weight="balanced"
    ),

    "RandomForest_bestIDM": RandomForestClassifier(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=300,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced_subsample"
    ),

    "ExtraTrees_bestIDM": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=200,
        n_jobs=-1,
        random_state=SEED,
        class_weight="balanced"
    ),

    "HistGB_bestIDM": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=SEED
    )
}

In [17]:
# ============================================================
# TRAIN BEST-IDM SELECTORS AND EVALUATE DOWNSTREAM IDM
# ============================================================

best_selector_overall_rows = []
best_selector_regime_rows = []
best_classification_rows = []

X_train_best = train_best[selector_cols]
y_train_best = train_best["best_idm_label"]

X_test_best = test_best[selector_cols]
y_test_best = test_best["best_idm_label"]

for name, clf in best_selector_models.items():
    print(f"\n==============================")
    print(f"Training selector: {name}")
    print(f"==============================")

    clf.fit(X_train_best, y_train_best)

    pred_test = clf.predict(X_test_best)

    print("\nBest-IDM label classification report on test:")
    print(classification_report(y_test_best, pred_test, digits=4))

    tmp = test_best.copy()
    tmp["selector_label"] = pred_test
    tmp = selected_idm_from_label(tmp, "selector_label")

    overall_df, regime_df = evaluate_idm_prediction(
        tmp,
        pred_col="a_idm_selected",
        model_name=name
    )

    best_selector_overall_rows.append(overall_df)
    best_selector_regime_rows.append(regime_df)

    best_classification_rows.append({
        "Model": name,
        "Accuracy": np.mean(pred_test == y_test_best),
        "Pred_accelerating": np.sum(pred_test == "accelerating"),
        "Pred_braking": np.sum(pred_test == "braking"),
        "Pred_global": np.sum(pred_test == "global"),
        "Pred_steady": np.sum(pred_test == "steady")
    })


best_selector_overall_df = pd.concat(best_selector_overall_rows, ignore_index=True)
best_selector_regime_df = pd.concat(best_selector_regime_rows, ignore_index=True)
best_classification_df = pd.DataFrame(best_classification_rows)

print("\n=== BEST-IDM SELECTOR DOWNSTREAM IDM OVERALL RESULTS ===")
print(
    best_selector_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


Training selector: DecisionTree_bestIDM

Best-IDM label classification report on test:
              precision    recall  f1-score   support

accelerating     0.4581    0.5305    0.4917    492715
     braking     0.3901    0.4593    0.4218    353701
      global     0.2949    0.4802    0.3654    445596
      steady     0.7461    0.4544    0.5648   1076162

    accuracy                         0.4758   2368174
   macro avg     0.4723    0.4811    0.4609   2368174
weighted avg     0.5481    0.4758    0.4907   2368174


Training selector: RandomForest_bestIDM

Best-IDM label classification report on test:
              precision    recall  f1-score   support

accelerating     0.4621    0.5224    0.4904    492715
     braking     0.3786    0.4810    0.4237    353701
      global     0.3024    0.4845    0.3724    445596
      steady     0.7536    0.4537    0.5664   1076162

    accuracy                         0.4779   2368174
   macro avg     0.4742    0.4854    0.4632   2368174
weighted 

In [18]:
# ============================================================
# COMBINE OVERALL RESULTS
# ============================================================

all_overall_df = pd.concat(
    [
        baseline_overall_df,
        regime_selector_overall_df,
        best_selector_overall_df
    ],
    ignore_index=True
)

all_regime_df = pd.concat(
    [
        baseline_regime_df,
        regime_selector_regime_df,
        best_selector_regime_df
    ],
    ignore_index=True
)

print("\n=== ALL IDM SELECTOR OVERALL RESULTS ===")
print(
    all_overall_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


=== ALL IDM SELECTOR OVERALL RESULTS ===
                  Model      MAE     RMSE    MedAE        R2      Bias      Corr
Ideal_BestCandidate_IDM 0.108056 0.195589 0.069797  0.637827 -0.006277  0.804899
  TrueRegime_Oracle_IDM 0.129256 0.214151 0.092586  0.565826  0.001369  0.753399
          HistGB_regime 0.219493 0.330417 0.145608 -0.033596 -0.018659  0.150472
         HistGB_bestIDM 0.224295 0.336129 0.148338 -0.069640  0.017877  0.175893
             KNN_regime 0.228113 0.342685 0.151161 -0.111770 -0.008887  0.119426
     ExtraTrees_bestIDM 0.246545 0.359692 0.169059 -0.224861  0.019856  0.149367
   DecisionTree_bestIDM 0.247561 0.360555 0.170697 -0.230746  0.025939  0.145625
   RandomForest_bestIDM 0.248426 0.361639 0.170555 -0.238156  0.016335  0.144212
      ExtraTrees_regime 0.279725 0.401954 0.199442 -0.529605 -0.008181  0.137049
    RandomForest_regime 0.282082 0.403320 0.201817 -0.540014 -0.006157  0.131353
    DecisionTree_regime 0.303043 0.422089 0.231073 -0.686685 -0.036

In [19]:
# ============================================================
# XGBOOST BEST-IDM SELECTOR: HARD AND SOFT MIXTURE
# ============================================================

le_best = LabelEncoder()

y_train_xgb = le_best.fit_transform(train_best["best_idm_label"])
y_test_xgb  = le_best.transform(test_best["best_idm_label"])

xgb_best = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

xgb_best.fit(train_best[selector_cols], y_train_xgb)

# -------------------------
# Hard selection
# -------------------------
pred_idx = xgb_best.predict(test_best[selector_cols])
pred_label = le_best.inverse_transform(pred_idx.astype(int))

xgb_hard_tmp = test_best.copy()
xgb_hard_tmp["selector_label"] = pred_label
xgb_hard_tmp = selected_idm_from_label(xgb_hard_tmp, "selector_label")

xgb_hard_overall, xgb_hard_regime = evaluate_idm_prediction(
    xgb_hard_tmp,
    pred_col="a_idm_selected",
    model_name="XGB_bestIDM_hard"
)

# -------------------------
# Soft mixture
# -------------------------
proba = xgb_best.predict_proba(test_best[selector_cols])

xgb_soft_tmp = test_best.copy()
xgb_soft_tmp["a_idm_soft"] = 0.0

# Store XGB probabilities and hard label for later analysis/export
for j, cls in enumerate(le_best.classes_):
    xgb_soft_tmp[f"p_{cls}"] = proba[:, j]

hard_idx = np.argmax(proba, axis=1)
xgb_soft_tmp["xgb_best_idm_hard_label"] = le_best.inverse_transform(hard_idx)

class_to_col = {
    "global": "a_idm_global",
    "braking": "a_idm_braking",
    "accelerating": "a_idm_accel",
    "steady": "a_idm_steady"
}

for j, cls in enumerate(le_best.classes_):
    xgb_soft_tmp["a_idm_soft"] += proba[:, j] * xgb_soft_tmp[class_to_col[cls]].values

xgb_soft_overall, xgb_soft_regime = evaluate_idm_prediction(
    xgb_soft_tmp,
    pred_col="a_idm_soft",
    model_name="XGB_bestIDM_soft"
)

print("\n=== XGBOOST BEST-IDM HARD VS SOFT ===")
print(
    pd.concat([xgb_hard_overall, xgb_soft_overall], ignore_index=True)[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].to_string(index=False)
)

print("\n=== XGBOOST BEST-IDM BY REGIME ===")
print(
    pd.concat([xgb_hard_regime, xgb_soft_regime], ignore_index=True)[
        ["Model", "Regime", "MAE", "RMSE", "Bias", "Count"]
    ].sort_values(["Regime", "MAE"]).to_string(index=False)
)


=== XGBOOST BEST-IDM HARD VS SOFT ===
           Model      MAE     RMSE    MedAE        R2      Bias     Corr
XGB_bestIDM_hard 0.224091 0.336030 0.148077 -0.069008  0.017487 0.176342
XGB_bestIDM_soft 0.205977 0.315965 0.132582  0.054845 -0.007289 0.237547

=== XGBOOST BEST-IDM BY REGIME ===
           Model       Regime      MAE     RMSE      Bias   Count
XGB_bestIDM_hard accelerating 0.302498 0.401468 -0.278115  672329
XGB_bestIDM_soft accelerating 0.330939 0.427213 -0.330722  672329
XGB_bestIDM_soft      braking 0.495501 0.578292  0.494717  284097
XGB_bestIDM_hard      braking 0.521790 0.637621  0.520584  284097
XGB_bestIDM_soft       steady 0.088202 0.115115  0.045720 1411748
XGB_bestIDM_hard       steady 0.126843 0.175614  0.057022 1411748


In [20]:
# ============================================================
# UPDATE ALL RESULTS WITH XGBOOST
# ============================================================

all_overall_with_xgb_df = pd.concat(
    [
        all_overall_df,
        xgb_hard_overall,
        xgb_soft_overall
    ],
    ignore_index=True
)

all_regime_with_xgb_df = pd.concat(
    [
        all_regime_df,
        xgb_hard_regime,
        xgb_soft_regime
    ],
    ignore_index=True
)

print("\n=== ALL IDM SELECTOR OVERALL RESULTS INCLUDING XGBOOST ===")
print(
    all_overall_with_xgb_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)

print("\n=== TOP 5 IDM SELECTORS PER REGIME INCLUDING XGBOOST ===")
best_per_regime_xgb = (
    all_regime_with_xgb_df
    .sort_values("MAE")
    .groupby("Regime")
    .head(5)
    .reset_index(drop=True)
)

print(
    best_per_regime_xgb[
        ["Regime", "Model", "MAE", "RMSE", "Bias", "Count"]
    ].to_string(index=False)
)


=== ALL IDM SELECTOR OVERALL RESULTS INCLUDING XGBOOST ===
                  Model      MAE     RMSE    MedAE        R2      Bias      Corr
Ideal_BestCandidate_IDM 0.108056 0.195589 0.069797  0.637827 -0.006277  0.804899
  TrueRegime_Oracle_IDM 0.129256 0.214151 0.092586  0.565826  0.001369  0.753399
       XGB_bestIDM_soft 0.205977 0.315965 0.132582  0.054845 -0.007289  0.237547
          HistGB_regime 0.219493 0.330417 0.145608 -0.033596 -0.018659  0.150472
       XGB_bestIDM_hard 0.224091 0.336030 0.148077 -0.069008  0.017487  0.176342
         HistGB_bestIDM 0.224295 0.336129 0.148338 -0.069640  0.017877  0.175893
             KNN_regime 0.228113 0.342685 0.151161 -0.111770 -0.008887  0.119426
     ExtraTrees_bestIDM 0.246545 0.359692 0.169059 -0.224861  0.019856  0.149367
   DecisionTree_bestIDM 0.247561 0.360555 0.170697 -0.230746  0.025939  0.145625
   RandomForest_bestIDM 0.248426 0.361639 0.170555 -0.238156  0.016335  0.144212
      ExtraTrees_regime 0.279725 0.401954 0.19944

In [21]:
# ============================================================
# XGBOOST TRUE-REGIME SELECTOR: HARD AND SOFT MIXTURE
# ============================================================

le_regime = LabelEncoder()

y_train_xgb_regime = le_regime.fit_transform(train_sel["true_regime"])
y_test_xgb_regime  = le_regime.transform(test_sel["true_regime"])

xgb_regime = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

xgb_regime.fit(train_sel[selector_cols], y_train_xgb_regime)

# -------------------------
# Hard regime selection
# -------------------------
pred_idx = xgb_regime.predict(test_sel[selector_cols])
pred_regime = le_regime.inverse_transform(pred_idx.astype(int))

xgb_regime_hard_tmp = test_sel.copy()
xgb_regime_hard_tmp["selector_label"] = pred_regime
xgb_regime_hard_tmp = selected_idm_from_label(
    xgb_regime_hard_tmp,
    "selector_label"
)

xgb_regime_hard_overall, xgb_regime_hard_regime = evaluate_idm_prediction(
    xgb_regime_hard_tmp,
    pred_col="a_idm_selected",
    model_name="XGB_regime_hard"
)

print("\n=== XGBOOST REGIME CLASSIFICATION REPORT ===")
print(classification_report(
    test_sel["true_regime"],
    pred_regime,
    digits=4
))

# -------------------------
# Soft regime mixture
# -------------------------
proba = xgb_regime.predict_proba(test_sel[selector_cols])

xgb_regime_soft_tmp = test_sel.copy()
xgb_regime_soft_tmp["a_idm_soft"] = 0.0

regime_to_col = {
    "braking": "a_idm_braking",
    "accelerating": "a_idm_accel",
    "steady": "a_idm_steady"
}

for j, cls in enumerate(le_regime.classes_):
    xgb_regime_soft_tmp["a_idm_soft"] += (
        proba[:, j] * xgb_regime_soft_tmp[regime_to_col[cls]].values
    )

xgb_regime_soft_overall, xgb_regime_soft_regime = evaluate_idm_prediction(
    xgb_regime_soft_tmp,
    pred_col="a_idm_soft",
    model_name="XGB_regime_soft"
)

print("\n=== XGBOOST REGIME HARD VS SOFT IDM ===")
print(
    pd.concat(
        [xgb_regime_hard_overall, xgb_regime_soft_overall],
        ignore_index=True
    )[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].to_string(index=False)
)

print("\n=== XGBOOST REGIME HARD VS SOFT BY REGIME ===")
print(
    pd.concat(
        [xgb_regime_hard_regime, xgb_regime_soft_regime],
        ignore_index=True
    )[
        ["Model", "Regime", "MAE", "RMSE", "Bias", "Count"]
    ].sort_values(["Regime", "MAE"]).to_string(index=False)
)


=== XGBOOST REGIME CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

accelerating     0.4951    0.2137    0.2985    672329
     braking     0.5450    0.1533    0.2392    284097
      steady     0.6445    0.9122    0.7553   1411748

    accuracy                         0.6228   2368174
   macro avg     0.5615    0.4264    0.4310   2368174
weighted avg     0.5901    0.6228    0.5637   2368174


=== XGBOOST REGIME HARD VS SOFT IDM ===
          Model      MAE     RMSE    MedAE        R2      Bias     Corr
XGB_regime_hard 0.219634 0.330531 0.145914 -0.034310 -0.018571 0.150078
XGB_regime_soft 0.210453 0.322840 0.134823  0.013263  0.000563 0.146873

=== XGBOOST REGIME HARD VS SOFT BY REGIME ===
          Model       Regime      MAE     RMSE      Bias   Count
XGB_regime_soft accelerating 0.330528 0.435423 -0.330208  672329
XGB_regime_hard accelerating 0.343459 0.430443 -0.332028  672329
XGB_regime_hard      braking 0.506620 0.612602  0.502615  284097
XGB_regim

In [22]:
# ============================================================
# HISTGB TRUE-REGIME SELECTOR: SOFT MIXTURE
# ============================================================

histgb_regime_soft_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.03,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=SEED
)

histgb_regime_soft_model.fit(
    train_sel[selector_cols],
    train_sel["true_regime"]
)

# Hard
histgb_regime_pred = histgb_regime_soft_model.predict(test_sel[selector_cols])

histgb_regime_hard_tmp = test_sel.copy()
histgb_regime_hard_tmp["selector_label"] = histgb_regime_pred
histgb_regime_hard_tmp = selected_idm_from_label(
    histgb_regime_hard_tmp,
    "selector_label"
)

histgb_regime_hard_overall, histgb_regime_hard_regime = evaluate_idm_prediction(
    histgb_regime_hard_tmp,
    pred_col="a_idm_selected",
    model_name="HistGB_regime_hard_tuned"
)

# Soft
histgb_proba = histgb_regime_soft_model.predict_proba(test_sel[selector_cols])

histgb_regime_soft_tmp = test_sel.copy()
histgb_regime_soft_tmp["a_idm_soft"] = 0.0

for j, cls in enumerate(histgb_regime_soft_model.classes_):
    histgb_regime_soft_tmp["a_idm_soft"] += (
        histgb_proba[:, j] * histgb_regime_soft_tmp[regime_to_col[cls]].values
    )

histgb_regime_soft_overall, histgb_regime_soft_regime = evaluate_idm_prediction(
    histgb_regime_soft_tmp,
    pred_col="a_idm_soft",
    model_name="HistGB_regime_soft"
)

print("\n=== HISTGB REGIME HARD VS SOFT IDM ===")
print(
    pd.concat(
        [histgb_regime_hard_overall, histgb_regime_soft_overall],
        ignore_index=True
    )[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].to_string(index=False)
)


=== HISTGB REGIME HARD VS SOFT IDM ===
                   Model      MAE     RMSE    MedAE        R2      Bias     Corr
HistGB_regime_hard_tuned 0.219490 0.330252 0.145667 -0.032565 -0.018582 0.150814
      HistGB_regime_soft 0.210508 0.322970 0.134744  0.012468  0.000642 0.145125


In [23]:
# ============================================================
# FINAL EXTENDED COMBINATION OF ALL RESULTS
# ============================================================

all_overall_extended_df = pd.concat(
    [
        all_overall_with_xgb_df,
        xgb_regime_hard_overall,
        xgb_regime_soft_overall,
        histgb_regime_hard_overall,
        histgb_regime_soft_overall
    ],
    ignore_index=True
)

all_regime_extended_df = pd.concat(
    [
        all_regime_with_xgb_df,
        xgb_regime_hard_regime,
        xgb_regime_soft_regime,
        histgb_regime_hard_regime,
        histgb_regime_soft_regime
    ],
    ignore_index=True
)

print("\n=== FINAL EXTENDED IDM SELECTOR OVERALL RESULTS ===")
print(
    all_overall_extended_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)


=== FINAL EXTENDED IDM SELECTOR OVERALL RESULTS ===
                   Model      MAE     RMSE    MedAE        R2      Bias      Corr
 Ideal_BestCandidate_IDM 0.108056 0.195589 0.069797  0.637827 -0.006277  0.804899
   TrueRegime_Oracle_IDM 0.129256 0.214151 0.092586  0.565826  0.001369  0.753399
        XGB_bestIDM_soft 0.205977 0.315965 0.132582  0.054845 -0.007289  0.237547
         XGB_regime_soft 0.210453 0.322840 0.134823  0.013263  0.000563  0.146873
      HistGB_regime_soft 0.210508 0.322970 0.134744  0.012468  0.000642  0.145125
HistGB_regime_hard_tuned 0.219490 0.330252 0.145667 -0.032565 -0.018582  0.150814
           HistGB_regime 0.219493 0.330417 0.145608 -0.033596 -0.018659  0.150472
         XGB_regime_hard 0.219634 0.330531 0.145914 -0.034310 -0.018571  0.150078
        XGB_bestIDM_hard 0.224091 0.336030 0.148077 -0.069008  0.017487  0.176342
          HistGB_bestIDM 0.224295 0.336129 0.148338 -0.069640  0.017877  0.175893
              KNN_regime 0.228113 0.342685 0.

In [24]:
# ============================================================
# BEST MODEL PER REGIME - FINAL EXTENDED TABLE
# ============================================================

print("\n=== FINAL EXTENDED IDM SELECTOR RESULTS BY REGIME ===")
print(
    all_regime_extended_df[
        ["Model", "Regime", "MAE", "RMSE", "Bias", "Count"]
    ].sort_values(["Regime", "MAE"]).to_string(index=False)
)

best_per_regime_extended = (
    all_regime_extended_df
    .sort_values("MAE")
    .groupby("Regime")
    .head(5)
    .reset_index(drop=True)
)

print("\n=== FINAL TOP 5 IDM SELECTORS PER REGIME BY MAE ===")
print(
    best_per_regime_extended[
        ["Regime", "Model", "MAE", "RMSE", "Bias", "Count"]
    ].to_string(index=False)
)


=== FINAL EXTENDED IDM SELECTOR RESULTS BY REGIME ===
                   Model       Regime      MAE     RMSE      Bias   Count
 Ideal_BestCandidate_IDM accelerating 0.145086 0.271497 -0.090415  672329
   TrueRegime_Oracle_IDM accelerating 0.169303 0.280187 -0.047553  672329
    DecisionTree_bestIDM accelerating 0.299023 0.428596 -0.270019  672329
      ExtraTrees_bestIDM accelerating 0.302155 0.428723 -0.275658  672329
          HistGB_bestIDM accelerating 0.302465 0.401208 -0.278016  672329
        XGB_bestIDM_hard accelerating 0.302498 0.401468 -0.278115  672329
    RandomForest_bestIDM accelerating 0.306826 0.433980 -0.280222  672329
      HistGB_regime_soft accelerating 0.330514 0.435463 -0.330202  672329
         XGB_regime_soft accelerating 0.330528 0.435423 -0.330208  672329
        XGB_bestIDM_soft accelerating 0.330939 0.427213 -0.330722  672329
         XGB_regime_hard accelerating 0.343459 0.430443 -0.332028  672329
HistGB_regime_hard_tuned accelerating 0.343467 0.429837 -

In [25]:
# ============================================================
# FINAL CONSISTENCY CHECK
# ============================================================

print("BRAKE_THRESHOLD:", BRAKE_THRESHOLD)
print("ACCEL_THRESHOLD:", ACCEL_THRESHOLD)
print("RUN_DIR:", RUN_DIR)

print("\nFinal overall table shape:", all_overall_extended_df.shape)
print("Final regime table shape :", all_regime_extended_df.shape)

print("\nModels included in final overall table:")
for model_name in sorted(all_overall_extended_df["Model"].unique()):
    print("-", model_name)

print("\nFinal overall results sorted by MAE:")
print(
    all_overall_extended_df[
        ["Model", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].sort_values("MAE").to_string(index=False)
)

BRAKE_THRESHOLD: -0.2
ACCEL_THRESHOLD: 0.2
RUN_DIR: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_symmetric_brake0p2_accel0p2

Final overall table shape: (18, 7)
Final regime table shape : (54, 9)

Models included in final overall table:
- DecisionTree_bestIDM
- DecisionTree_regime
- ExtraTrees_bestIDM
- ExtraTrees_regime
- Global_IDM
- HistGB_bestIDM
- HistGB_regime
- HistGB_regime_hard_tuned
- HistGB_regime_soft
- Ideal_BestCandidate_IDM
- KNN_regime
- RandomForest_bestIDM
- RandomForest_regime
- TrueRegime_Oracle_IDM
- XGB_bestIDM_hard
- XGB_bestIDM_soft
- XGB_regime_hard
- XGB_regime_soft

Final overall results sorted by MAE:
                   Model      MAE     RMSE    MedAE        R2      Bias      Corr
 Ideal_BestCandidate_IDM 0.108056 0.195589 0.069797  0.637827 -0.006277  0.804899
   TrueRegime_Oracle_IDM 0.129256 0.214151 0.092586  0.565826  0.001369  0.753399
        XGB_bestIDM_soft 0.2059

In [26]:
# ============================================================
# SAVE FINAL EXTENDED RESULTS
# ============================================================

all_overall_extended_df.to_csv(
    RUN_DIR / "idm_selector_overall_results.csv",
    index=False
)

all_regime_extended_df.to_csv(
    RUN_DIR / "idm_selector_regime_results.csv",
    index=False
)

regime_classification_df.to_csv(
    RUN_DIR / "regime_selector_classification_summary.csv",
    index=False
)

best_classification_df.to_csv(
    RUN_DIR / "best_idm_selector_classification_summary.csv",
    index=False
)

joblib.dump(
    {
        "model": xgb_best,
        "label_encoder": le_best,
        "selector_cols": selector_cols,
        "classes": list(le_best.classes_),
        "regime_tag": REGIME_TAG,
        "threshold_tag": THRESHOLD_TAG,
        "best_global_params": best_global_params,
        "best_params_braking": best_params_braking,
        "best_params_accel": best_params_accel,
        "best_params_steady": best_params_steady,
    },
    RUN_DIR / "xgb_soft_best_idm_selector_bundle.joblib"
)

print("Saved XGB-soft best-IDM selector bundle:")
print(RUN_DIR / "xgb_soft_best_idm_selector_bundle.joblib")
print("Saved final extended results to:")
print(RUN_DIR)

Saved XGB-soft best-IDM selector bundle:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_symmetric_brake0p2_accel0p2\xgb_soft_best_idm_selector_bundle.joblib
Saved final extended results to:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_symmetric_brake0p2_accel0p2


In [27]:
# ============================================================
# SAVE XGB-SOFT BEST-IDM TEST PREDICTIONS
# ============================================================

xgb_soft_predictions = xgb_soft_tmp.copy()

xgb_soft_predictions = xgb_soft_predictions.rename(columns={
    "a_true": "y_true",
    "a_idm_soft": "y_pred"
})

xgb_soft_predictions["Model"] = "XGB-soft best-IDM selector"
xgb_soft_predictions["error"] = (
    xgb_soft_predictions["y_pred"] - xgb_soft_predictions["y_true"]
)

xgb_soft_predictions.to_csv(
    RUN_DIR / "xgb_soft_best_idm_test_predictions.csv",
    index=False
)

print("Saved XGB-soft IDM test predictions to:")
print(RUN_DIR / "xgb_soft_best_idm_test_predictions.csv")

Saved XGB-soft IDM test predictions to:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\idm_selector_comparison_runs\idm_selector_comparison_symmetric_brake0p2_accel0p2\xgb_soft_best_idm_test_predictions.csv
